# Baseline ResNet-18 - Nordland nhận dạng địa điểm

ResNet-18 ImageNet **đóng băng** (không train), GeM pooling trên layer4, chuẩn hoá L2, so cosine.
Notebook gồm: tự đánh giá Recall@1/5 trên đoạn cuối của train (val tự chia), rồi dự đoán test và ghi `submission.csv`.

Mô hình chưa học gì về đổi mùa nên điểm rất thấp. Hướng cải thiện: fine-tune trên các cặp đông-hè trong `train.csv`
(contrastive / triplet / ArcFace theo vị trí), thử layer3, tăng độ phân giải, dùng thêm thứ tự khung của database khi so khớp.

In [ ]:
import os, random, time
from pathlib import Path

import numpy as np
import torch, torchvision
import torchvision.transforms as T
import pandas as pd
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# Kaggle: tự tìm thư mục dữ liệu có query.csv trong /kaggle/input; chạy trên máy: đặt DATA trỏ tới thư mục dữ liệu
tim = sorted(Path("/kaggle/input").rglob("query.csv")) if Path("/kaggle/input").exists() else []
D = tim[0].parent if tim else Path(os.environ.get("DATA", "upload"))
DEV = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 2 if os.name != "nt" else 0
print(D, DEV)

In [ ]:
class GeM(torch.nn.Module):
    def __init__(self, p=3.0):
        super().__init__(); self.p = p
    def forward(self, x):
        return x.clamp(min=1e-6).pow(self.p).mean((2, 3)).pow(1 / self.p)

backbone = torchvision.models.resnet18(weights="DEFAULT")
model = torch.nn.Sequential(*list(backbone.children())[:-2], GeM()).eval().to(DEV)
tf = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class Anh(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return tf(Image.open(self.paths[i]).convert("RGB"))

def dac_trung(paths, bs=128):
    dl = torch.utils.data.DataLoader(Anh(paths), batch_size=bs, num_workers=WORKERS)
    out = []
    with torch.inference_mode():
        for x in dl:
            out.append(torch.nn.functional.normalize(model(x.to(DEV)), dim=1).cpu())
    return torch.cat(out).numpy()

## Tự đánh giá trên val: 2.000 khung cuối của train (đông làm truy vấn, hè làm database)

Đúng nếu ảnh tìm được lệch tối đa 2 khung so với vị trí thật, giống cách chấm trên leaderboard.

In [ ]:
tr = pd.read_csv(D / "train.csv")
val = tr.tail(2000).reset_index(drop=True)
t0 = time.time()
Qv = dac_trung([D / p for p in val.winter_path]); Bv = dac_trung([D / p for p in val.summer_path])
top = np.argsort(-(Qv @ Bv.T), axis=1)[:, :5]
dung = np.abs(top - np.arange(len(val))[:, None]) <= 2
print(f"val Recall@1 = {dung[:, 0].mean():.4f}   Recall@5 = {dung.any(1).mean():.4f}   ({time.time() - t0:.0f}s)")

## Dự đoán test và ghi submission.csv (10 ảnh database khớp nhất cho mỗi truy vấn)

In [ ]:
q, db = pd.read_csv(D / "query.csv"), pd.read_csv(D / "database.csv")
Q = dac_trung([D / f"test/query/{i}.jpg" for i in q.id])
B = dac_trung([D / f"test/database/{i}.jpg" for i in db.id])
top = np.argsort(-(Q @ B.T), axis=1)[:, :10]
sub = pd.DataFrame({"id": q.id, "predictions": [" ".join(db.id.values[t]) for t in top]})
sub.to_csv("submission.csv", index=False)
sub.head()